In [4]:
from pathlib import Path
import re

import pandas as pd

PROJECT_ROOT = Path("../data")

RAW_DIR = PROJECT_ROOT / "raw"
PROCESSED_DIR = PROJECT_ROOT / "processed"

OUTPUT_FILE = PROCESSED_DIR / "ev_registrations_historical.csv"

def clean_number(value):
    """
    Convert Statistik Austria number strings to integers.

    Examples:
    "175,909" -> 175909
    "100.0"   -> 100.0, but this script only uses absolute columns
    "-"       -> None
    """
    if pd.isna(value):
        return None

    text = str(value).strip()

    if text in {"", "-", "–"}:
        return None

    text = text.replace("\xa0", "")
    text = text.replace(" ", "")

    # In the absolute-count columns, Statistik Austria uses comma as thousands separator.
    text = text.replace(",", "")

    return int(float(text))

def read_raw_file(file_path):
    """
    Read a historical Statistik Austria raw file without assuming a clean header.

    The historical sheet has two header rows:
    - row 1: period groups, e.g. 1. Halbjahr 2019
    - row 2: metrics, e.g. absolut, Anteil in %, Veränderung in %

    Therefore we read with header=None.
    """
    suffix = file_path.suffix.lower()

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(file_path, header=None, dtype=str)

    if suffix in {".csv", ".txt", ".tsv"}:
        for encoding in ["utf-8-sig", "utf-8", "latin1"]:
            for sep in ["\t", ";", ","]:
                try:
                    df = pd.read_csv(
                        file_path,
                        header=None,
                        dtype=str,
                        sep=sep,
                        encoding=encoding,
                        engine="python",
                    )

                    if df.shape[1] > 3:
                        return df

                except Exception:
                    continue

    raise ValueError(f"Could not read file: {file_path}")

def find_row_containing(df, text, start_row=0):
    """
    Return the first row index where any cell contains the search text.
    """
    search = text.lower()

    for idx in range(start_row, len(df)):
        row_text = " ".join(df.iloc[idx].dropna().astype(str)).lower()
        if search in row_text:
            return idx

    raise ValueError(f"Could not find row containing: {text}")


def extract_year_columns(df, period_header_row, metric_header_row):
    """
    Detect columns for year groups.

    Example structure:

    row period_header_row:
    Kfz-Art | 1. Halbjahr 2019 | NaN | NaN | 1. Halbjahr 2020 | NaN | NaN ...

    row metric_header_row:
            | absolut | Anteil in % | Veränderung in % | absolut | Anteil in % ...

    We only want each year's 'absolut' column.
    """
    year_columns = {}

    current_year = None

    for col_idx in range(df.shape[1]):
        period_value = df.iat[period_header_row, col_idx]
        metric_value = df.iat[metric_header_row, col_idx]

        if pd.notna(period_value):
            period_text = str(period_value).strip()
            year_match = re.search(r"1\.\s*Halbjahr\s*(20\d{2}|19\d{2})", period_text)

            if year_match:
                current_year = int(year_match.group(1))

        if current_year is None:
            continue

        metric_text = "" if pd.isna(metric_value) else str(metric_value).strip().lower()

        if metric_text == "absolut":
            year_columns[current_year] = col_idx

    if not year_columns:
        raise ValueError("Could not detect any '1. Halbjahr YYYY / absolut' columns.")

    return year_columns


def find_section_bounds(df, section_title):
    """
    Find the start and end row of a section.

    A section starts with a row like:
    - Kfz insgesamt
    - Darunter alternative Antriebe
    - Darunter Elektro-Antrieb (BEV)

    The section ends before the next section-like row.
    """
    start_idx = find_row_containing(df, section_title)

    section_markers = [
        "Kfz insgesamt",
        "Darunter alternative Antriebe",
        "Darunter Elektro-Antrieb",
        "Q:",
    ]

    end_idx = len(df)

    for idx in range(start_idx + 1, len(df)):
        first_cell = df.iat[idx, 0]

        if pd.isna(first_cell):
            continue

        first_cell_text = str(first_cell).strip()

        for marker in section_markers:
            if marker.lower() in first_cell_text.lower():
                end_idx = idx
                return start_idx, end_idx

    return start_idx, end_idx


def extract_personenkraftwagen_values(df, section_title, year_columns):
    """
    Extract Personenkraftwagen absolute values from a named section.

    For this project:
    - section 'Kfz insgesamt' gives total passenger-car registrations
    - section 'Darunter Elektro-Antrieb (BEV)' gives electric passenger-car registrations
    """
    start_idx, end_idx = find_section_bounds(df, section_title)

    section = df.iloc[start_idx + 1:end_idx].copy()

    personenkraftwagen_rows = section[
        section.iloc[:, 0].astype(str).str.strip().str.lower() == "personenkraftwagen"
    ]

    if personenkraftwagen_rows.empty:
        raise ValueError(f"Could not find Personenkraftwagen row in section: {section_title}")

    row = personenkraftwagen_rows.iloc[0]

    values = {}

    for year, col_idx in year_columns.items():
        values[year] = clean_number(row.iloc[col_idx])

    return values


def process_historical_file(file_path):
    """
    Process one historical H1 file into normalized rows.

    Output is one row per year:
    - month is set to YYYY-06-01 because the value represents Jan-Jun cumulative data.
    """
    df = read_raw_file(file_path)

    period_header_row = find_row_containing(df, "Kfz-Art")
    metric_header_row = period_header_row + 1

    year_columns = extract_year_columns(
        df=df,
        period_header_row=period_header_row,
        metric_header_row=metric_header_row,
    )

    total_values = extract_personenkraftwagen_values(
        df=df,
        section_title="Kfz insgesamt",
        year_columns=year_columns,
    )

    electric_values = extract_personenkraftwagen_values(
        df=df,
        section_title="Darunter Elektro-Antrieb",
        year_columns=year_columns,
    )

    rows = []

    for year in sorted(year_columns):
        total_new_registrations = total_values.get(year)
        electric_new_registrations = electric_values.get(year)

        if total_new_registrations is None:
            raise ValueError(f"Missing total registrations for {year} in {file_path.name}")

        if electric_new_registrations is None:
            raise ValueError(f"Missing electric registrations for {year} in {file_path.name}")

        if electric_new_registrations > total_new_registrations:
            raise ValueError(
                f"Invalid data for {year}: electric registrations exceed total registrations."
            )

        ev_share = electric_new_registrations / total_new_registrations

        rows.append(
            {
                "month": f"{year}-06-01",
                "period_type": "H1",
                "period_start": f"{year}-01-01",
                "period_end": f"{year}-06-30",
                "total_new_registrations": total_new_registrations,
                "electric_new_registrations": electric_new_registrations,
                "ev_share": ev_share,
                "source_file": file_path.name,
            }
        )

    return rows

def find_candidate_files():
    """
    Find likely historical files.

    Keep this conservative. We do not process every raw file automatically because
    recent Statistik Austria files have a different structure.
    """
    patterns = [
        "*historical*.*",
        "*historisch*.*",
        "*Historisch*.*",
        "*halbjahr*.*",
        "*Halbjahr*.*",
        "*2019*2023*.*",
    ]

    files = []

    for pattern in patterns:
        files.extend(RAW_DIR.glob(pattern))

    files = sorted(set(files))

    if not files:
        raise FileNotFoundError(
            "No candidate historical files found in data/raw.\n"
            "Rename the historical file so its filename contains one of:\n"
            "- historical\n"
            "- historisch\n"
            "- Halbjahr\n"
            "- 2019_2023"
        )

    return files

In [5]:
def main():
    candidate_files = find_candidate_files()

    all_rows = []

    for file_path in candidate_files:
        print(f"Inspecting candidate file: {file_path.name}")

        try:
            rows = process_historical_file(file_path)
            all_rows.extend(rows)
            print(f"Processed: {file_path.name}")

        except Exception as error:
            print(f"Skipped: {file_path.name}")
            print(f"Reason: {error}")

    if not all_rows:
        raise RuntimeError("No historical rows were processed successfully.")

    output = pd.DataFrame(all_rows)

    output["month"] = pd.to_datetime(output["month"])
    output["period_start"] = pd.to_datetime(output["period_start"])
    output["period_end"] = pd.to_datetime(output["period_end"])

    output = output.sort_values(["month", "source_file"])

    duplicate_months = output[output.duplicated("month", keep=False)]

    if not duplicate_months.empty:
        print("\nWarning: duplicate months found. Review before using this file:")
        print(duplicate_months)

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    output.to_csv(OUTPUT_FILE, index=False)

    print("\nSaved historical processed data:")
    print(OUTPUT_FILE)

    print("\nPreview:")
    print(output)


if __name__ == "__main__":
    main()

Inspecting candidate file: neuzulassungen_pkw_2024_halbjahr_1.ods
Skipped: neuzulassungen_pkw_2024_halbjahr_1.ods
Reason: Could not read file: ../data/raw/neuzulassungen_pkw_2024_halbjahr_1.ods


RuntimeError: No historical rows were processed successfully.